# Stabilizer formalism: the Steane $[[7,1,3]]$ code

We use the symplectic Pauli implementation in `qec_project.codes.pauli` and the Steane construction in `qec_project.codes.steane` to inspect stabilizers, logical operators, and a handful of error syndromes.

In [1]:
from qec_project.codes.pauli import Pauli, commutes_with_set
from qec_project.codes.steane import (
    steane_logical_x,
    steane_logical_z,
    steane_stabilizers,
    verify_steane,
)

print('verify_steane():', verify_steane())

verify_steane(): True


## Stabilizer table

Three X-type and three Z-type generators, each of weight 4.

In [2]:
stabs = steane_stabilizers()
header = f"{'idx':>3} | {'type':>4} | {'operator':>10} | {'wt':>2}"
print(header)
print('-' * len(header))
for i, s in enumerate(stabs):
    kind = 'X' if s.x.any() else 'Z'
    print(f'{i:>3} | {kind:>4} | {s.to_string():>10} | {s.weight():>2}')

idx | type |   operator | wt
----------------------------
  0 |    X |   +XIXIXIX |  4
  1 |    X |   +IXXIIXX |  4
  2 |    X |   +IIIXXXX |  4
  3 |    Z |   +ZIZIZIZ |  4
  4 |    Z |   +IZZIIZZ |  4
  5 |    Z |   +IIIZZZZ |  4


## Logical operators

`X_L` and `Z_L` each have weight 3 (a minimum-weight representative within their coset of the stabilizer group), and they anti-commute.

In [3]:
lx = steane_logical_x()
lz = steane_logical_z()
print('X_L =', lx.to_string(), '  weight', lx.weight())
print('Z_L =', lz.to_string(), '  weight', lz.weight())
print('[X_L, Z_L] commute?', lx.commutes_with(lz))

X_L = +IXIXIXI   weight 3
Z_L = +IZIZIZI   weight 3
[X_L, Z_L] commute? False


## Syndromes of a few sample errors

Each row records which of the six stabilizers anti-commutes with the error (a `1` means the stabilizer flags the error).

In [4]:
samples = [
    'IIIIIII',  # no error
    'XIIIIII',  # single X on qubit 0
    'IZIIIII',  # single Z on qubit 1
    'IIYIIII',  # single Y on qubit 2
    'XIZIIII',  # X on 0, Z on 2 (two-qubit)
    'IIIIIYI',  # single Y near the end
]
labels = [f'S{i}' for i in range(len(stabs))]
print('error    | ' + ' '.join(labels))
print('-' * (11 + 3 * len(labels)))
for s in samples:
    err = Pauli.from_string(s)
    bits = [0 if err.commutes_with(g) else 1 for g in stabs]
    print(f'{s} | ' + '  '.join(str(b) for b in bits))
print()
print('All sample errors detected by at least one stabilizer?')
non_trivial = [Pauli.from_string(s) for s in samples if any(ch != 'I' for ch in s)]
print(all(not commutes_with_set(e, stabs) for e in non_trivial))

error    | S0 S1 S2 S3 S4 S5
-----------------------------
IIIIIII | 0  0  0  0  0  0
XIIIIII | 0  0  0  1  0  0
IZIIIII | 0  1  0  0  0  0
IIYIIII | 1  1  0  1  1  0
XIZIIII | 1  1  0  1  0  0
IIIIIYI | 0  1  1  0  1  1

All sample errors detected by at least one stabilizer?
True


## Observation

Every weight-1 and weight-2 Pauli on seven qubits triggers at least one stabilizer, which is the algebraic statement that Steane has distance 3. The logical operators sit at exactly that minimum weight.